<a href="https://colab.research.google.com/github/SushanthMorsu/AI-Resume-Screener/blob/main/Task_2_Sentiment_Analysis_using_NLP_Pipeline_and_ML_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Import Libraries


In [1]:
import pandas as pd
import numpy as np
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

# 2. Load Dataset


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [4]:
df = pd.read_csv(r"/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")

print(df.head())
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


# 3. Data Understanding


In [5]:
print("Class Distribution:")
print(df['sentiment'].value_counts())

print("\nSample Reviews:")
print(df['review'][0])
print(df['review'][1])

# Convert labels into numbers
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Reviews:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings 

# 4. NLP Preprocessing

In [6]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):

    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r'http\S+|www\S+', '', text)

    # remove html tags
    text = re.sub(r'<.*?>', '', text)

    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # remove numbers
    text = re.sub(r'\d+', '', text)

    # tokenization
    tokens = word_tokenize(text)

    # remove stopwords + stemming
    words = []
    for word in tokens:
        if word not in stop_words:
            words.append(stemmer.stem(word))

    return " ".join(words)

# Apply preprocessing


In [8]:
import nltk
nltk.download('punkt_tab')
df['clean_review'] = df['review'].apply(clean_text)

print(df[['review','clean_review']].head())

# =====================================================
# 5. Train Test Split
# =====================================================

X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                        clean_review  
0  one review mention watch oz episod youll hook ...  
1  wonder littl product film techniqu unassum old...  
2  thought wonder way spend time hot summer weeke...  
3  basic there famili littl boy jake think there ...  
4  petter mattei love time money visual stun film...  


# 6. Feature Engineering


In [9]:
# -------- Bag of Words --------
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# -------- TF-IDF --------
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 7. Model Training Function


In [10]:
def evaluate_model(name, model, X_train, X_test):

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)
    pre = precision_score(y_test, pred)
    rec = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)

    print(f"\n{name}")
    print("Accuracy :", acc)
    print("Precision:", pre)
    print("Recall   :", rec)
    print("F1 Score :", f1)

    return [name, acc, pre, rec, f1]

# 8. Train Models on BoW


In [11]:
results = []

print("===== USING BAG OF WORDS =====")

results.append(evaluate_model(
    "Logistic Regression (BoW)",
    LogisticRegression(max_iter=1000),
    X_train_bow, X_test_bow
))

results.append(evaluate_model(
    "Naive Bayes (BoW)",
    MultinomialNB(),
    X_train_bow, X_test_bow
))

results.append(evaluate_model(
    "Decision Tree (BoW)",
    DecisionTreeClassifier(),
    X_train_bow, X_test_bow
))

===== USING BAG OF WORDS =====

Logistic Regression (BoW)
Accuracy : 0.8673
Precision: 0.86278342455043
Recall   : 0.8759674538598928
F1 Score : 0.8693254554406696

Naive Bayes (BoW)
Accuracy : 0.8435
Precision: 0.8511928831378892
Recall   : 0.8354832307997618
F1 Score : 0.8432648973460191

Decision Tree (BoW)
Accuracy : 0.7236
Precision: 0.7310582977859029
Recall   : 0.7142290136931931
F1 Score : 0.7225456735595261


# 9. Train Models on TF-IDF


In [12]:
print("\n===== USING TF-IDF =====")

results.append(evaluate_model(
    "Logistic Regression (TFIDF)",
    LogisticRegression(max_iter=1000),
    X_train_tfidf, X_test_tfidf
))

results.append(evaluate_model(
    "Naive Bayes (TFIDF)",
    MultinomialNB(),
    X_train_tfidf, X_test_tfidf
))

results.append(evaluate_model(
    "Decision Tree (TFIDF)",
    DecisionTreeClassifier(),
    X_train_tfidf, X_test_tfidf
))


===== USING TF-IDF =====

Logistic Regression (TFIDF)
Accuracy : 0.8833
Precision: 0.8717357910906298
Recall   : 0.9009724151617384
F1 Score : 0.8861130086854689

Naive Bayes (TFIDF)
Accuracy : 0.846
Precision: 0.8427032321253672
Recall   : 0.8537408215915856
F1 Score : 0.848186119873817

Decision Tree (TFIDF)
Accuracy : 0.7212
Precision: 0.7261402451275869
Recall   : 0.7172057948005557
F1 Score : 0.7216453674121406


# 10. Comparison Table


In [13]:
results_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1 Score"
])

print("\nFinal Comparison:")
print(results_df.sort_values(by="F1 Score", ascending=False))



Final Comparison:
                         Model  Accuracy  Precision    Recall  F1 Score
3  Logistic Regression (TFIDF)    0.8833   0.871736  0.900972  0.886113
0    Logistic Regression (BoW)    0.8673   0.862783  0.875967  0.869325
4          Naive Bayes (TFIDF)    0.8460   0.842703  0.853741  0.848186
1            Naive Bayes (BoW)    0.8435   0.851193  0.835483  0.843265
2          Decision Tree (BoW)    0.7236   0.731058  0.714229  0.722546
5        Decision Tree (TFIDF)    0.7212   0.726140  0.717206  0.721645


##     Summary:

1. Best Model:
Usually Logistic Regression + TF-IDF gives highest accuracy.

2. Best Vectorization:
TF-IDF performs better than BoW because it gives weight to important words.

3. Naive Bayes:
Fast and simple baseline model.

4. Decision Tree:
Easy to interpret but may overfit.

5. Best Preprocessing Steps:
Lowercasing, punctuation removal, stopwords removal, stemming.

Pipeline:
Raw Text -> Cleaning -> Vectorization -> ML Model -> Prediction